# v0.27.0 — Subqueries

## 1. Intro

Since v0.27.0 a QuerySet can filter another: `Subquery(qs)`, or the queryset itself, is the value of a
lookup in `filter()` or `Q`. The ORM sends **one request**: the server runs the inner queryset inside
the outer statement, so both read the same state (one consistent read), and the inner queryset's values
are bound like any filter value.

The name and the import are the full SurrealDB-ORM's (`from surreal_orm_lite import Subquery`). The ORM
behaves the same on SurrealDB 2.x and 3.x: this notebook runs top to bottom on either, and only the cell
that prints the server line differs.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


In [2]:
client = await SurrealDBConnectionManager.get_client()
version = str(await client.version())  # "surrealdb-<major>.<minor>.<patch>"
line = int(version.removeprefix("surrealdb-").split(".")[0])
print(f"SurrealDB {line}.x")

SurrealDB 3.x


## 3. Models and data

People and their posts: a post's `author` is a record link to a person. Post `five` has no author.

The notebook starts by dropping its tables, so it can be re-executed in place; on SurrealDB 3.x removing
a missing table raises, hence the suppressed errors.

In [3]:
from surrealdb import RecordID

from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict


class Person(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_person")

    id: str | None = None
    name: str
    active: bool = True
    age: int = 0


class Post(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_post")

    id: str | None = None
    title: str
    author: RecordID | None = None  # a record link to a Person
    score: int = 0


DEMO_TABLES = ["nb_person", "nb_post", "nb_ghost"]


async def reset() -> None:
    """Drop every table of this notebook, whether or not it exists."""
    for table in DEMO_TABLES:
        with contextlib.suppress(Exception):
            await client.query(f"REMOVE TABLE {table};", {})


def titles(posts: list[Post]) -> list[str]:
    """The titles of the posts, sorted."""
    return sorted(post.title for post in posts)


def keys(people: list[Person]) -> list[str]:
    """The ids of the people, sorted."""
    return sorted(person.id or "" for person in people)


await reset()

for key, name, is_active, age in [("ada", "Ada", True, 36), ("bob", "Bob", True, 50), ("cy", "Cy", False, 20)]:
    await Person(id=key, name=name, active=is_active, age=age).save()
for key, title, author, score in [
    ("p1", "one", "ada", 5),
    ("p2", "two", "bob", 8),
    ("p3", "three", "cy", 2),
    ("p4", "four", "ada", 9),
    ("p5", "five", None, 1),  # no author
]:
    link = RecordID(Person.get_table_name(), author) if author else None
    await Post(id=key, title=title, author=link, score=score).save()

print("people:", [(p.id, p.name, p.active, p.age) for p in await Person.objects().order_by("id").exec()])
print("posts: ", [(p.title, p.author.id if p.author else None, p.score) for p in await Post.objects().order_by("id").exec()])

people: [('ada', 'Ada', True, 36), ('bob', 'Bob', True, 50), ('cy', 'Cy', False, 20)]
posts:  [('one', 'ada', 5), ('two', 'bob', 8), ('three', 'cy', 2), ('four', 'ada', 9), ('five', None, 1)]


## 4. `in` and `not_in`

`active` is a queryset that is never run on its own: the outer queryset reads it when it runs, in the
same statement. Given bare, a queryset is read as `Subquery(queryset)`.

`not_in` keeps post `five`: a post without an author has no value in the list of active people.

QuerySet methods change the queryset they are called on, and a subquery is read when the outer queryset
runs: a filter added to `active` later would apply to every read that uses it afterwards. Build a new
`Model.objects()` for another subquery.

In [4]:
from surreal_orm_lite import Subquery

active = Person.objects().filter(active=True)

print("author__in=Subquery(active):", titles(await Post.objects().filter(author__in=Subquery(active)).exec()))
print("author__in=active:          ", titles(await Post.objects().filter(author__in=active).exec()))
print("author__not_in=active:      ", titles(await Post.objects().filter(author__not_in=active).exec()))
print("author__not_in=active count:", await Post.objects().filter(author__not_in=active).count())

author__in=Subquery(active): ['four', 'one', 'two']
author__in=active:           ['four', 'one', 'two']
author__not_in=active:       ['five', 'three']
author__not_in=active count: 2


## 5. What a subquery selects

The inner queryset gives the outer filter **one list of values**: without `select()`, the ids of its
records (`SELECT VALUE id`); with `select("field")`, the values of that one field. Here a subquery gives
the record links the posts hold, and that subquery is itself inside another one.

Values are compared as stored, never converted: to match records, select `id` or a record link (a
`"table:key"` string never equals a record id).

In [5]:
top_authors = Post.objects().filter(score__gt=8).select("author")
print("people who wrote a post scoring > 8:", keys(await Person.objects().filter(id__in=top_authors).exec()))

wrote_a_top_post = Person.objects().filter(id__in=top_authors)
print("posts by those people:              ", titles(await Post.objects().filter(author__in=wrote_a_top_post).exec()))

people who wrote a post scoring > 8: ['ada']
posts by those people:               ['four', 'one']


## 6. Ordered and limited

`order_by()`, `limit()` and `offset()` choose the values a subquery gives: here, the oldest active person.

In [6]:
oldest = Person.objects().filter(active=True).order_by("-age").limit(1)
print("posts of the oldest active person:", titles(await Post.objects().filter(author__in=oldest).exec()))

posts of the oldest active person: ['two']


A statement with subqueries is one **block**, whose `LET`s bind each subquery once, innermost first. The
read above is sent as:

```sql
{ LET $_sq0 = (SELECT id AS _sqv, age FROM nb_person WHERE active = $_f0 ORDER BY age DESC LIMIT 1)._sqv; (SELECT * FROM nb_post WHERE author IN $_sq0) };
```

* **Why a `LET`**: SurrealDB 3.2 and later can return wrong rows for an inline sub-SELECT with `ORDER BY`
  and `LIMIT`, which gives its value to the first outer record only
  ([surrealdb/surrealdb#7470](https://github.com/surrealdb/surrealdb/issues/7470)). Bound once, the
  subquery is right on every line, and the server runs it once instead of once per record.
* **Why `_sqv`**: an ordered subquery selects its column as `_sqv`, with its order fields, and reads
  `._sqv` back: SurrealDB 2.x refuses to order a sub-SELECT of one column by another field.
* `$_sq0`, `$_sq1`, … are reserved for these `LET`s.

## 7. Compared with the first value

`exact` (no suffix), `gt`, `gte`, `lt` and `lte` compare the field with the subquery's **first** value,
and `contains`/`not_contains` look for it in an array field: order the subquery to choose it.

A first-value lookup on an empty subquery matches **no record**, and `~Q(...)` of it every record.
SurrealDB alone would compare with `array::first([])`, which is `NONE`, and `score > NONE` holds for
every record, so the ORM guards the comparison (`$_sq0 != [] AND …`).

In [7]:
from surreal_orm_lite import Q

best = Post.objects().select("score").order_by("-score").limit(1)
print("score=Subquery(best):            ", titles(await Post.objects().filter(score=Subquery(best)).exec()))

nothing = Post.objects().filter(score__gt=100).select("score")
print("score__gt=Subquery(nothing):     ", titles(await Post.objects().filter(score__gt=Subquery(nothing)).exec()))
print("~Q(score__gt=Subquery(nothing)): ", titles(await Post.objects().filter(~Q(score__gt=Subquery(nothing))).exec()))

score=Subquery(best):             ['four']
score__gt=Subquery(nothing):      []
~Q(score__gt=Subquery(nothing)):  ['five', 'four', 'one', 'three', 'two']


## 8. With `Q` and aggregates

Subqueries combine with `Q` (`&`, `|`, `~`), and every read honours them: `count()`, `sum()`, `avg()`,
`min()`, `max()`, `values(…).annotate(…)`, `exists()`, `first()`…

In [8]:
from surreal_orm_lite import Count

inactive = Person.objects().filter(active=False)
either = Post.objects().filter(Q(author__in=inactive) | Q(score__gte=9))
print("by an inactive person, or scoring >= 9:", titles(await either.exec()))

print("posts by active people, count(): ", await Post.objects().filter(author__in=active).count())
print("posts by active people, sum():   ", await Post.objects().filter(author__in=active).sum("score"))

groups = await Post.objects().filter(author__in=active).values("author").annotate(n=Count()).exec()
print("posts per active author:         ", sorted((row["author"].id, row["n"]) for row in groups))

by an inactive person, or scoring >= 9: ['four', 'three']
posts by active people, count():  3
posts by active people, sum():    22
posts per active author:          [('ada', 2), ('bob', 1)]


## 9. Writes by filter

`bulk_update()`, `bulk_delete()` and `QuerySet.patch()` honour a subquery in their filter, and so do the
lookups of `get_or_create()` and `update_or_create()`. Here the posts of inactive people get a score of
0, then `get_or_create()` finds post `one` by a subquery lookup instead of creating a record.

In [9]:
updated = await Post.objects().filter(author__in=inactive).bulk_update(score=0)
print("bulk_update(score=0):", updated, "post updated")
print("scores:", sorted((post.title, post.score) for post in await Post.objects().exec()))

post, created = await Post.objects().get_or_create(author__in=active, title="one")
print("get_or_create(author__in=active, title='one'):", post.id, "created" if created else "found")

bulk_update(score=0): 1 post updated
scores: [('five', 1), ('four', 9), ('one', 5), ('three', 0), ('two', 8)]
get_or_create(author__in=active, title='one'): p1 found


## 10. What a subquery refuses

A subquery gives one list of values, and it is a filter value, never data. These raise when the outer
queryset compiles, before any request (only the message up to its first colon is printed):

* `select()` of several fields (a list of objects would match nothing), `values()` and `annotate()`;
* a lookup other than the eleven that compare with its values (`in`, `not_in`, `containsall`, `containsany`,
  `exact`, `gt`, `gte`, `lt`, `lte`, `contains`, `not_contains`): `isnull`, `like`, `startswith`, `inside`, …;
* a subquery written as a value: `bulk_update(score=Subquery(...))`.

In [10]:
from surreal_orm_lite import SurrealDbError

attempts = {
    'select("id", "name") in a subquery': lambda: (
        Post.objects().filter(author__in=Person.objects().select("id", "name")).exec()
    ),
    "isnull with a subquery": lambda: Post.objects().filter(author__isnull=Subquery(active)).exec(),
    "bulk_update(score=Subquery(best))": lambda: (
        Post.objects().filter(author__in=inactive).bulk_update(score=Subquery(best))
    ),
}
for label, attempt in attempts.items():
    try:
        await attempt()
    except (ValueError, TypeError, SurrealDbError) as error:
        print(f"{label}\n    {type(error).__name__}: {str(error).split(':')[0]}")

select("id", "name") in a subquery
    ValueError: A subquery cannot honour select() of several fields
isnull with a subquery
    ValueError: 'isnull' cannot take a subquery
bulk_update(score=Subquery(best))
    ValueError: bulk_update() writes its values as data; 'score' got a subquery, which is a filter value


## 11. A subquery table that does not exist

The ORM reads a missing table as empty, a subquery's table too. SurrealDB 2.x reads it so in the same
request. SurrealDB 3.x raises: the ORM then asks the server whether the table exists and, once it confirms
it does not, sends the statement again with that subquery empty. The answer is the same on both lines.

In [11]:
class Ghost(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_ghost")  # never written

    id: str | None = None
    x: int = 0


print("author__not_in=Ghost.objects():", titles(await Post.objects().filter(author__not_in=Ghost.objects()).exec()))
print("author__in=Ghost.objects():    ", titles(await Post.objects().filter(author__in=Ghost.objects()).exec()))

author__not_in=Ghost.objects(): ['five', 'four', 'one', 'three', 'two']
author__in=Ghost.objects():     []


## 12. Cleanup

Drop the demo tables and close the connection.

In [12]:
await reset()
await SurrealDBConnectionManager.close_connection()
print("Done")

Done
